# Подготовка данных: Amazon, Retailrocket и REES46

Получаем каталог товаров и связи `buy` / `view`. У Amazon связи уже есть в metadata;
у Retailrocket и REES46 строим их из событий. Совместная покупка — лишь приближённый признак дополняемости.

```text
project/data/
├── raw/
│   ├── amazon/meta_Musical_Instruments.json.gz
│   ├── retailrocket/
│   │   ├── events.csv
│   │   ├── item_properties_part1.csv
│   │   └── item_properties_part2.csv
│   ├── rees46_cosmetics/
│   └── rees46_electronics/
└── processed/
    ├── amazon/
    ├── retailrocket/
    ├── rees46_cosmetics/
    └── rees46_electronics/
```

## 1. Скачивание датасетов

In [34]:
%pip install -q pandas kagglehub


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [35]:
import gzip
import json
from itertools import combinations
from pathlib import Path

import pandas as pd


ROOT = Path.cwd()
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed"

### 1.1. Amazon: metadata Musical Instruments

Скачиваем `.json.gz` по ссылке с [официальной страницы Amazon 2018](https://cseweb.ucsd.edu/~jmcauley/datasets/amazon_v2/)

In [36]:
import shutil
from urllib.request import urlopen

amazon_path = RAW / "amazon" / "meta_Musical_Instruments.json.gz"
amazon_url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/metaFiles2/meta_Musical_Instruments.json.gz"
amazon_path.parent.mkdir(parents=True, exist_ok=True)

if amazon_path.exists() and amazon_path.stat().st_size > 0:
    print("Amazon уже скачан:", amazon_path)

else:
    temporary = amazon_path.with_suffix(".gz.part")
    
    print("Скачиваем Amazon…")
    try:
        with urlopen(amazon_url, timeout=120) as response, temporary.open("wb") as f:
            shutil.copyfileobj(response, f)
        temporary.replace(amazon_path)
    finally:
        temporary.unlink(missing_ok=True)
    
    print("Сохранено:", amazon_path)

Скачиваем Amazon…
Сохранено: /Users/karamikha/Documents/HSE/RecSys/project/data/raw/amazon/meta_Musical_Instruments.json.gz


### 1.2. Retailrocket: события и свойства товаров

Скачиваем три CSV через [kagglehub](https://github.com/Kaggle/kagglehub)

Источник: [Retailrocket recommender system dataset на Kaggle](https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset).

In [37]:
import kagglehub

retail_dir = RAW / "retailrocket"
retail_dir.mkdir(parents=True, exist_ok=True)
retail_files = [
    "events.csv", "item_properties_part1.csv",
    "item_properties_part2.csv",
]

for name in retail_files:
    path = retail_dir / name
    if path.exists() and path.stat().st_size > 0:
        print("Уже скачан:", name)
        continue
    
    print("Скачиваем:", name)
    kagglehub.dataset_download(
        "retailrocket/ecommerce-dataset", path=name, output_dir=str(retail_dir),
    )
    
print("Retailrocket готов:", retail_dir)

Скачиваем: events.csv


100%|██████████| 32.4M/32.4M [00:04<00:00, 8.00MB/s]

Extracting zip of events.csv...


Скачиваем: item_properties_part1.csv


100%|██████████| 140M/140M [00:15<00:00, 9.37MB/s] 


Extracting zip of item_properties_part1.csv...
Скачиваем: item_properties_part2.csv


100%|██████████| 118M/118M [00:20<00:00, 5.96MB/s] 

Extracting zip of item_properties_part2.csv...


Retailrocket готов: /Users/karamikha/Documents/HSE/RecSys/project/data/raw/retailrocket


**Общая функция скачивания двух наборов REES46 через Kaggle.**

In [38]:
def download_rees(name, handle):
    """Скачивает REES46 в папку набора; повторные загрузки контролирует kagglehub."""
    folder = RAW / name
    folder.mkdir(parents=True, exist_ok=True)
    kagglehub.dataset_download(handle, output_dir=str(folder))
    print("Готово:", folder)

### 1.3. REES46 Cosmetics Events

Папка: `data/raw/rees46_cosmetics/`.

Источник: [Cosmetics Events на Kaggle](https://www.kaggle.com/datasets/mkechinov/ecommerce-events-history-in-cosmetics-shop).

In [39]:
download_rees("rees46_cosmetics", "mkechinov/ecommerce-events-history-in-cosmetics-shop")

100%|██████████| 430M/430M [00:59<00:00, 7.58MB/s] 

Extracting files...


Готово: /Users/karamikha/Documents/HSE/RecSys/project/data/raw/rees46_cosmetics


### 1.4. REES46 Electronics Events

Папка: `data/raw/rees46_electronics/`.

Источник: [Electronics Events на Kaggle](https://www.kaggle.com/datasets/mkechinov/ecommerce-events-history-in-electronics-store).

In [40]:
download_rees("rees46_electronics", "mkechinov/ecommerce-events-history-in-electronics-store")

100%|██████████| 20.3M/20.3M [00:02<00:00, 7.88MB/s]

Extracting files...


Готово: /Users/karamikha/Documents/HSE/RecSys/project/data/raw/rees46_electronics


## 2. Обработка датасетов

### 2.1. Amazon Reviews 2018 — Musical Instruments

In [41]:
amazon_path = RAW / "amazon" / "meta_Musical_Instruments.json.gz"

In [42]:
from itertools import islice


with gzip.open(amazon_path, "rt", encoding="utf-8") as f:
    raw_sample = [json.loads(line) for line in islice(f, 5)]
print("Amazon — исходные данные:")
display(pd.DataFrame(raw_sample))

Amazon — исходные данные:


,category,tech1,description,fit,title,also_buy,tech2,brand,feature,rank,also_view,main_cat,similar_item,date,price,asin,imageURL,imageURLHighRes
0,"[Musical Instruments, Drums & Percussion, Hand...",,[Cricket Rubbing the spine with the wooden sti...,,Wooden Percussion 2 Piece Set of 3 Inch Cricke...,"[B00NP8GYVS, B00NP80XMO, B00NP8M098]",,WADSUWAN SHOP,"[Wood percussion, Owl whistle*, Includes woode...","[>#141,729 in Musical Instruments (See Top 100...",[],Musical Instruments,,"December 2, 2013",,0000989983,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...
1,"[Musical Instruments, Drums & Percussion, Hand...",,[Frog - Rubbing its spine with the wooden stic...,,"Wooden Percussion 3 Piece Set Frog, Cricket an...","[B00NP8GYVS, B00NP80XMO, B01MY48HK5, B00AZZ1AJ...",,WADSUWAN SHOP,"[Wood percussion, Small 3 inches, Creates orig...","[>#1,622 in Musical Instruments (See Top 100 i...",[],Musical Instruments,,"December 2, 2013",$0.91,0000098906,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...
2,"[Musical Instruments, Instrument Accessories, ...",,[Vivaldi's famous set of four violin concertos...,,Hal Leonard Vivaldi Four Seasons for Piano (Or...,[],,Hal Leonard,"[., ., .]","[>#330,653 in Musical Instruments (See Top 100...",[],Musical Instruments,,"May 10, 2011",$62.93,0041291905,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...
3,[],,"[The Turn of the Screw (op. 54) vocal score, p...",,The Turn of the Screw (vocal score),"[0486266842, 0793507669, 0393008789, 142341280...",,Boosey &amp; Hawkes,[],"[>#86,354 in Musical Instruments (See Top 100 ...",[],Musical Instruments,,"May 23, 2007",$107.79,0060015500,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...
4,[],,[],,Suite for Organ (including the Trumpet Volunta...,[],,,[],"[>#482,025 in Musical Instruments (See Top 100...",[],Musical Instruments,,"February 8, 2013",,0193757710,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...


In [43]:
records = []
with gzip.open(amazon_path, "rt", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        categories = row["category"]
        records.append({
            "item_id": row["asin"],
            "title": row.get("title"),
            "description": row.get("description"),
            "category_path": " > ".join(categories),
            "category": categories[-1] if categories else None,
            "buy": row.get("also_buy") or [],
            "view": row["also_view"],
        })

amazon = pd.DataFrame(records)
print("Строк:", len(amazon), "Дубликатов ID:", amazon.item_id.duplicated().sum())
amazon = amazon.drop_duplicates("item_id").reset_index(drop=True)
amazon_items = amazon.drop(columns=["buy", "view"])

print("Amazon — после обработки: товары")
display(amazon_items.head())

Строк: 120310 Дубликатов ID: 8175
Amazon — после обработки: товары


,item_id,title,description,category_path,category
0,0000989983,Wooden Percussion 2 Piece Set of 3 Inch Cricke...,[Cricket Rubbing the spine with the wooden sti...,Musical Instruments > Drums & Percussion > Han...,Guiros
1,0000098906,"Wooden Percussion 3 Piece Set Frog, Cricket an...",[Frog - Rubbing its spine with the wooden stic...,Musical Instruments > Drums & Percussion > Han...,Sound Effects
2,0041291905,Hal Leonard Vivaldi Four Seasons for Piano (Or...,[Vivaldi's famous set of four violin concertos...,Musical Instruments > Instrument Accessories >...,General Accessories
3,0060015500,The Turn of the Screw (vocal score),"[The Turn of the Screw (op. 54) vocal score, p...",,None
4,0193757710,Suite for Organ (including the Trumpet Volunta...,[],,None


#### Готовые связи → таблица пар

Категория кандидата задаёт целевую категорию запроса.

In [44]:
amazon_parts = []
for relation in ["buy", "view"]:
    pairs = amazon[["item_id", relation]].explode(relation)
    pairs.columns = ["source_id", "target_id"]
    pairs = pairs.dropna().drop_duplicates()
    pairs = pairs[pairs.source_id.ne(pairs.target_id)]
    before = len(pairs)
    pairs = pairs[pairs.target_id.isin(amazon_items.item_id)]
    print(relation, "вне каталога:", before - len(pairs))

    amazon_parts.append(pairs.assign(relation=relation))

amazon_pairs = pd.concat(amazon_parts, ignore_index=True)
categories = amazon_items.set_index("item_id")["category"]
amazon_pairs["source_category"] = amazon_pairs.source_id.map(categories)
amazon_pairs["target_category"] = amazon_pairs.target_id.map(categories)
print("Пар без категории:", amazon_pairs[["source_category", "target_category"]].isna().any(axis=1).sum())
amazon_pairs = amazon_pairs.dropna(subset=["source_category", "target_category"])

display(amazon_pairs.groupby("relation").size().rename("pairs"))
print("Amazon — после обработки: пары")
display(amazon_pairs.head())

buy вне каталога: 116360
view вне каталога: 398389
Пар без категории: 13367


relation
buy      85710
view    244651
Name: pairs, dtype: int64

Amazon — после обработки: пары


,source_id,target_id,relation,source_category,target_category
0,0000989983,B00NP8GYVS,buy,Guiros,Guiros
1,0000989983,B00NP80XMO,buy,Guiros,Guiros
2,0000989983,B00NP8M098,buy,Guiros,Guiros
3,0000098906,B00NP8GYVS,buy,Sound Effects,Guiros
4,0000098906,B00NP80XMO,buy,Sound Effects,Guiros


### 2.2. Retailrocket recommender system dataset

Сессии определяем по паузе **более 30 минут**.

In [45]:
retail_dir = RAW / "retailrocket"
events = pd.read_csv(retail_dir / "events.csv", dtype={"transactionid": "Int64"})
print("Retailrocket — исходные события:")
display(events.head())

Retailrocket — исходные события:


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,<NA>
1,1433224214164,992329,view,248676,<NA>
2,1433221999827,111016,view,318965,<NA>
3,1433221955914,483717,view,253185,<NA>
4,1433221337106,951259,view,367447,<NA>


In [46]:
events = events.drop_duplicates().dropna(subset=["timestamp", "visitorid", "itemid", "event"])
events["timestamp"] = pd.to_datetime(events.timestamp, unit="ms", utc=True)
events = events.rename(columns={"visitorid": "user_id", "itemid": "item_id"})
events = events.sort_values(["user_id", "timestamp"]).reset_index(drop=True)

gap = events.groupby("user_id").timestamp.diff()
events["session_id"] = (gap.isna() | gap.gt(pd.Timedelta(minutes=30))).cumsum()
display(events.groupby("event").size().rename("events"))
print("Retailrocket — после очистки и выделения сессий:")
display(events.head())

event
addtocart        68966
transaction      22457
view           2664218
Name: events, dtype: int64

Retailrocket — после очистки и выделения сессий:


,timestamp,user_id,event,item_id,transactionid,session_id
0,2015-09-11 20:49:49.439000+00:00,0,view,285930,<NA>,1
1,2015-09-11 20:52:39.591000+00:00,0,view,357564,<NA>,1
2,2015-09-11 20:55:17.175000+00:00,0,view,67045,<NA>,1
3,2015-08-13 17:46:06.444000+00:00,1,view,72028,<NA>,2
4,2015-08-07 17:51:44.567000+00:00,2,view,325215,<NA>,3


#### Категории товаров

Используем последнюю известную категорию **на момент события**.

In [47]:
property_files = sorted(retail_dir.glob("item_properties*.csv"))
parts = []
for path in property_files:
    for i, chunk in enumerate(pd.read_csv(path, dtype={"property": "string", "value": "string"}, chunksize=500_000)):
        if i == 0:
            print(f"Retailrocket — исходные свойства: {path.name}")
            display(chunk.head())
        parts.append(chunk.loc[chunk.property.eq("categoryid"), ["timestamp", "itemid", "value"]])

category_history = pd.concat(parts, ignore_index=True).rename(
    columns={"itemid": "item_id", "value": "category", "timestamp": "category_time"}
)
category_history["category"] = pd.to_numeric(category_history.category, errors="coerce").astype("Int64")
category_history["category_time"] = pd.to_datetime(category_history.category_time, unit="ms", utc=True)
category_history = category_history.dropna().drop_duplicates(["item_id", "category_time"], keep="last")
events = pd.merge_asof(
    events.sort_values("timestamp"), category_history.sort_values("category_time"),
    left_on="timestamp", right_on="category_time", by="item_id", direction="backward",
)
print(f"События с категорией: {events.category.notna().mean():.1%}")
print("Retailrocket — после обработки: события с категориями")
display(events.head())

Retailrocket — исходные свойства: item_properties_part1.csv


,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
1,1441508400000,206783,888,1116713 960601 n277.200
2,1439089200000,395014,400,n552.000 639502 n720.000 424566
3,1431226800000,59481,790,n15360.000
4,1431831600000,156781,917,828513


Retailrocket — исходные свойства: item_properties_part2.csv


,timestamp,itemid,property,value
0,1433041200000,183478,561,769062
1,1439694000000,132256,976,n26.400 1135780
2,1435460400000,420307,921,1149317 1257525
3,1431831600000,403324,917,1204143
4,1435460400000,230701,521,769062


События с категорией: 76.2%
Retailrocket — после обработки: события с категориями


,timestamp,user_id,event,item_id,transactionid,session_id,category_time,category
0,2015-05-03 03:00:04.384000+00:00,693516,addtocart,297662,<NA>,867565,NaT,<NA>
1,2015-05-03 03:00:11.289000+00:00,829044,view,60987,<NA>,1037522,NaT,<NA>
2,2015-05-03 03:00:13.048000+00:00,652699,view,252860,<NA>,816723,NaT,<NA>
3,2015-05-03 03:00:24.154000+00:00,1125936,view,33661,<NA>,1408621,NaT,<NA>
4,2015-05-03 03:00:26.228000+00:00,693516,view,297662,<NA>,867565,NaT,<NA>


#### Строим co-buy и co-view

У покупок группа — транзакция, у просмотров — сессия (сессии с более чем 50 различными
просмотренными товарами исключаем целиком, чтобы ограничить квадратичный рост числа пар).

In [48]:
MAX_VIEW_ITEMS = 50

def make_pairs(frame, group_col, relation, max_items=None):
    """Строит неупорядоченные пары уникальных товаров внутри групп каждого пользователя.

    Сохраняет тип связи, категории и границы времени событий в группе.
    Если задан max_items, исключает группы, превышающие лимит уникальных товаров."""
    
    rows, skipped = [], 0
    for (user_id, group_id), group in frame.groupby(["user_id", group_col], sort=False):
        items = group.sort_values("timestamp").drop_duplicates("item_id", keep="last").sort_values("item_id")
        if max_items is not None and len(items) > max_items:
            skipped += 1
            continue
        values = list(items[["item_id", "category"]].itertuples(index=False, name=None))
        start, end = group.timestamp.min(), group.timestamp.max()
        for (source, source_cat), (target, target_cat) in combinations(values, 2):
            rows.append((user_id, group_id, start, end, source, target, relation, source_cat, target_cat))
            
    print(relation, "исключённых больших групп:", skipped)
    return pd.DataFrame(rows, columns=[
        "user_id", "group_id", "group_start", "group_end", "source_id", "target_id",
        "relation", "source_category", "target_category",
    ])

In [49]:
buys = events[events.event.eq("transaction")]
print("Покупок без transactionid:", buys.transactionid.isna().sum())

buy_pairs = make_pairs(buys.dropna(subset=["transactionid"]), "transactionid", "buy")
view_pairs = make_pairs(events[events.event.eq("view")], "session_id", "view", MAX_VIEW_ITEMS)
retail_pairs = pd.concat([buy_pairs, view_pairs], ignore_index=True)
print("Пар без категории:", retail_pairs[["source_category", "target_category"]].isna().any(axis=1).sum())

retail_pairs = retail_pairs.dropna(subset=["source_category", "target_category"])
display(retail_pairs.groupby("relation").agg(pairs=("source_id", "size"), users=("user_id", "nunique")))
print("Retailrocket — после обработки: пары")
display(retail_pairs.head())

Покупок без transactionid: 0
buy исключённых больших групп: 0
view исключённых больших групп: 311
Пар без категории: 613985


,pairs,users
relation,,
buy,7758,1493
view,1317240,168314


Retailrocket — после обработки: пары


,user_id,group_id,group_start,group_end,source_id,target_id,relation,source_category,target_category
340,894034,7641,2015-05-10 13:33:22.411000+00:00,2015-05-10 13:33:22.458000+00:00,36026,120050,buy,579,1598
341,894034,7641,2015-05-10 13:33:22.411000+00:00,2015-05-10 13:33:22.458000+00:00,36026,262813,buy,579,860
342,894034,7641,2015-05-10 13:33:22.411000+00:00,2015-05-10 13:33:22.458000+00:00,120050,262813,buy,1598,860
345,1392946,10374,2015-05-10 20:34:08.278000+00:00,2015-05-10 20:34:08.653000+00:00,241555,412622,buy,858,858
346,1340188,15320,2015-05-10 20:56:21.567000+00:00,2015-05-10 20:56:21.754000+00:00,274435,369447,buy,1244,48


### 2.3. REES46 Cosmetics Events

In [50]:
def read_rees(folder):
    """Объединяет CSV и CSV.gz набора REES46, очищает события и унифицирует имена столбцов.

    Сохраняет точность ID, переводит время в UTC и возвращает события по времени."""
    
    paths = sorted(folder.rglob("*.csv")) + sorted(folder.rglob("*.csv.gz"))
    columns = ["event_time", "event_type", "product_id", "category_id", "category_code",
               "brand", "price", "user_id", "user_session"]
    types = {"product_id": "Int64", "category_id": "Int64", "user_id": "Int64",
             "event_type": "string", "user_session": "string", "category_code": "string",
             "brand": "string", "price": "float32"}

    frames = []
    for path in paths:
        part = pd.read_csv(path, usecols=columns, dtype=types)
        print(f"{folder.name} — исходные события: {path.name}")
        display(part.head())

        frames.append(part)
    
    frame = pd.concat(frames, ignore_index=True)
    frame = frame.rename(columns={"event_time": "timestamp", "event_type": "event",
                                  "product_id": "item_id", "category_id": "category",
                                  "user_session": "session_id"})
    frame["timestamp"] = pd.to_datetime(frame.timestamp, utc=True, errors="coerce")
    frame["session_id"] = frame.session_id.str.strip().replace("", pd.NA)
    before = len(frame)
    frame = frame.drop_duplicates().dropna(subset=["timestamp", "event", "item_id", "user_id"])
    print("Удалено дублей / некорректных событий:", before - len(frame))
    print("Без сессии:", frame.session_id.isna().sum(), "Без категории:", frame.category.isna().sum())
    display(frame.groupby("event").size().rename("events"))

    frame = frame.sort_values("timestamp").reset_index(drop=True)
    print(f"{folder.name} — после обработки: события")
    display(frame.head())
    
    return frame


def rees_pairs(frame):
    """Строит buy- и view-пары внутри пользовательских сессий REES46.

    Исключает события без ID сессии, пары без категорий и слишком большие
    view-сессии. Возвращает пары с пользователями и границами времени событий."""

    valid = frame.dropna(subset=["session_id"])
    buy = make_pairs(valid[valid.event.eq("purchase")], "session_id", "buy")
    view = make_pairs(valid[valid.event.eq("view")], "session_id", "view", MAX_VIEW_ITEMS)
    pairs = pd.concat([buy, view], ignore_index=True)
    print("Пар без категории:", pairs[["source_category", "target_category"]].isna().any(axis=1).sum())
    pairs = pairs.dropna(subset=["source_category", "target_category"])
    display(pairs.groupby("relation").agg(pairs=("source_id", "size"), users=("user_id", "nunique")))

    return pairs

Co-buy — пары различных купленных товаров внутри `(user_id, session_id)`;
co-view — пары просмотренных товаров внутри той же группы. 

In [51]:
cosmetics_events = read_rees(RAW / "rees46_cosmetics")
cosmetics_pairs = rees_pairs(cosmetics_events)

print("REES46 Cosmetics — после обработки: пары")
display(cosmetics_pairs.head())

rees46_cosmetics — исходные события: 2019-Dec.csv


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-12-01 00:00:00 UTC,remove_from_cart,5712790,1487580005268456287,<NA>,f.o.x,6.270000,576802932,51d85cb0-897f-48d2-918b-ad63965c12dc
1,2019-12-01 00:00:00 UTC,view,5764655,1487580005411062629,<NA>,cnd,29.049999,412120092,8adff31e-2051-4894-9758-224bfa8aec18
2,2019-12-01 00:00:02 UTC,cart,4958,1487580009471148064,<NA>,runail,1.190000,494077766,c99a50e8-2fac-4c4d-89ec-41c05f114554
3,2019-12-01 00:00:05 UTC,view,5848413,1487580007675986893,<NA>,freedecor,0.790000,348405118,722ffea5-73c0-4924-8e8f-371ff8031af4
4,2019-12-01 00:00:07 UTC,view,5824148,1487580005511725929,<NA>,<NA>,5.560000,576005683,28172809-7e4a-45ce-bab0-5efa90117cd5


rees46_cosmetics — исходные события: 2019-Nov.csv


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:02 UTC,view,5802432,1487580009286598681,<NA>,<NA>,0.320000,562076640,09fafd6c-6c99-46b1-834f-33527f4de241
1,2019-11-01 00:00:09 UTC,cart,5844397,1487580006317032337,<NA>,<NA>,2.380000,553329724,2067216c-31b5-455d-a1cc-af0575a34ffb
2,2019-11-01 00:00:10 UTC,view,5837166,1783999064103190764,<NA>,pnb,22.219999,556138645,57ed222e-a54a-4907-9944-5a875c2d7f4f
3,2019-11-01 00:00:11 UTC,cart,5876812,1487580010100293687,<NA>,jessnail,3.160000,564506666,186c1951-8052-4b37-adce-dd9644b1d5f7
4,2019-11-01 00:00:24 UTC,remove_from_cart,5826182,1487580007483048900,<NA>,<NA>,3.330000,553329724,2067216c-31b5-455d-a1cc-af0575a34ffb


rees46_cosmetics — исходные события: 2019-Oct.csv


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,cart,5773203,1487580005134238553,<NA>,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
1,2019-10-01 00:00:03 UTC,cart,5773353,1487580005134238553,<NA>,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
2,2019-10-01 00:00:07 UTC,cart,5881589,2151191071051219817,<NA>,lovely,13.48,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9
3,2019-10-01 00:00:07 UTC,cart,5723490,1487580005134238553,<NA>,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
4,2019-10-01 00:00:15 UTC,cart,5881449,1487580013522845895,<NA>,lovely,0.56,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9


... (8 outputs left out)

buy исключённых больших групп: 0
view исключённых больших групп: 3209
Пар без категории: 0


,pairs,users
relation,,
buy,11521008,98558
view,23839097,424331


REES46 Cosmetics — после обработки: пары


,user_id,group_id,group_start,group_end,source_id,target_id,relation,source_category,target_category
0,474232307,445f2b74-5e4c-427e-b7fa-6e0a28b156fe,2019-10-01 00:06:35+00:00,2019-10-01 00:06:35+00:00,5619862,5619866,buy,1487580006895846315,1487580006895846315
1,474232307,445f2b74-5e4c-427e-b7fa-6e0a28b156fe,2019-10-01 00:06:35+00:00,2019-10-01 00:06:35+00:00,5619862,5696184,buy,1487580006895846315,1487580005134238553
2,474232307,445f2b74-5e4c-427e-b7fa-6e0a28b156fe,2019-10-01 00:06:35+00:00,2019-10-01 00:06:35+00:00,5619862,5706784,buy,1487580006895846315,1487580005092295511
3,474232307,445f2b74-5e4c-427e-b7fa-6e0a28b156fe,2019-10-01 00:06:35+00:00,2019-10-01 00:06:35+00:00,5619862,5723503,buy,1487580006895846315,1487580005134238553
4,474232307,445f2b74-5e4c-427e-b7fa-6e0a28b156fe,2019-10-01 00:06:35+00:00,2019-10-01 00:06:35+00:00,5619862,5773207,buy,1487580006895846315,1487580005134238553


### 2.4. REES46 Electronics Events

In [52]:
electronics_events = read_rees(RAW / "rees46_electronics")
electronics_pairs = rees_pairs(electronics_events)

print("REES46 Electronics — после обработки: пары")
display(electronics_pairs.head())

rees46_electronics — исходные события: events.csv


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2020-09-24 11:57:06 UTC,view,1996170,2144415922528452715,electronics.telephone,<NA>,31.900000,1515915625519388267,LJuJVLEjPT
1,2020-09-24 11:57:26 UTC,view,139905,2144415926932472027,computers.components.cooler,zalman,17.160000,1515915625519380411,tdicluNnRY
2,2020-09-24 11:57:27 UTC,view,215454,2144415927158964449,<NA>,<NA>,9.810000,1515915625513238515,4TMArHtXQy
3,2020-09-24 11:57:33 UTC,view,635807,2144415923107266682,computers.peripherals.printer,pantum,113.809998,1515915625519014356,aGFYrNgC08
4,2020-09-24 11:57:36 UTC,view,3658723,2144415921169498184,<NA>,cameronsino,15.870000,1515915625510743344,aa4mmk0kwQ


Удалено дублей / некорректных событий: 655
Без сессии: 162 Без категории: 0


event
cart         54032
purchase     37343
view        793099
Name: events, dtype: int64

rees46_electronics — после обработки: события


,timestamp,event,item_id,category,category_code,brand,price,user_id,session_id
0,2020-09-24 11:57:06+00:00,view,1996170,2144415922528452715,electronics.telephone,<NA>,31.900000,1515915625519388267,LJuJVLEjPT
1,2020-09-24 11:57:26+00:00,view,139905,2144415926932472027,computers.components.cooler,zalman,17.160000,1515915625519380411,tdicluNnRY
2,2020-09-24 11:57:27+00:00,view,215454,2144415927158964449,<NA>,<NA>,9.810000,1515915625513238515,4TMArHtXQy
3,2020-09-24 11:57:33+00:00,view,635807,2144415923107266682,computers.peripherals.printer,pantum,113.809998,1515915625519014356,aGFYrNgC08
4,2020-09-24 11:57:36+00:00,view,3658723,2144415921169498184,<NA>,cameronsino,15.870000,1515915625510743344,aa4mmk0kwQ


buy исключённых больших групп: 0
view исключённых больших групп: 15
Пар без категории: 0


,pairs,users
relation,,
buy,5517,1598
view,431927,61234


REES46 Electronics — после обработки: пары


,user_id,group_id,group_start,group_end,source_id,target_id,relation,source_category,target_category
0,1515915625519356010,3jFpdbozOd,2020-09-24 12:29:49+00:00,2020-09-24 12:29:49+00:00,888783,1283197,buy,2144415922167742561,2144415973413749245
1,1515915625519045356,rOuRLFpG6U,2020-09-24 15:36:39+00:00,2020-09-24 15:40:39+00:00,575364,1703412,buy,2144415921421156430,2144415925196030129
2,1515915625519495480,UL6gfSiAth,2020-09-24 19:18:00+00:00,2020-09-24 19:18:00+00:00,3648406,3758770,buy,2144415922427789416,2144415924491387038
3,1515915625519495480,UL6gfSiAth,2020-09-24 19:18:00+00:00,2020-09-24 19:18:00+00:00,3648406,4013167,buy,2144415922427789416,2144415925196030129
4,1515915625519495480,UL6gfSiAth,2020-09-24 19:18:00+00:00,2020-09-24 19:18:00+00:00,3648406,4154233,buy,2144415922427789416,2144415926966026460


## 3. Сохранение обработанных данных

Корневая папка результатов: `data/processed/`.
Подпапки каждого датасета создаются автоматически при запуске ячейки сохранения.

Сохраняем только таблицы для дальнейшего разбиения и моделирования: у Amazon — `items` и `pairs`,
у остальных трёх наборов — `events` и `pairs`.

In [53]:
def save_tables(dataset, **tables):
    """Сохраняет таблицы одного датасета в его подпапку data/processed в формате CSV.gz."""
    folder = OUT / dataset
    folder.mkdir(parents=True, exist_ok=True)
    
    for name, table in tables.items():
        path = folder / f"{name}.csv.gz"
        table.to_csv(path, index=False)
        print(path, "строк:", len(table))

### 3.1. Amazon Musical Instruments

Папка: `data/processed/amazon/`. Файлы: `items.csv.gz`, `pairs.csv.gz`.

In [54]:
save_tables("amazon", items=amazon_items, pairs=amazon_pairs)

/Users/karamikha/Documents/HSE/RecSys/project/data/processed/amazon/items.csv.gz строк: 112135
/Users/karamikha/Documents/HSE/RecSys/project/data/processed/amazon/pairs.csv.gz строк: 330361


### 3.2. Retailrocket

Папка: `data/processed/retailrocket/`. Файлы: `events.csv.gz`, `pairs.csv.gz`.

In [55]:
save_tables("retailrocket", events=events, pairs=retail_pairs)

/Users/karamikha/Documents/HSE/RecSys/project/data/processed/retailrocket/events.csv.gz строк: 2755641
/Users/karamikha/Documents/HSE/RecSys/project/data/processed/retailrocket/pairs.csv.gz строк: 1324998


### 3.3. REES46 Cosmetics Events

Папка: `data/processed/rees46_cosmetics/`. Файлы: `events.csv.gz`, `pairs.csv.gz`.

In [56]:
save_tables("rees46_cosmetics", events=cosmetics_events, pairs=cosmetics_pairs)

/Users/karamikha/Documents/HSE/RecSys/project/data/processed/rees46_cosmetics/events.csv.gz строк: 19583742
/Users/karamikha/Documents/HSE/RecSys/project/data/processed/rees46_cosmetics/pairs.csv.gz строк: 35360105


### 3.4. REES46 Electronics Events

Папка: `data/processed/rees46_electronics/`. Файлы: `events.csv.gz`, `pairs.csv.gz`.

In [57]:
save_tables("rees46_electronics", events=electronics_events, pairs=electronics_pairs)

/Users/karamikha/Documents/HSE/RecSys/project/data/processed/rees46_electronics/events.csv.gz строк: 884474
/Users/karamikha/Documents/HSE/RecSys/project/data/processed/rees46_electronics/pairs.csv.gz строк: 437444
